# `aidp-epm-cloud` live test — HTTP Basic (default for v0.1)
**Live-test row 11.** Username MUST be in `tenancy.user@domain` form (e.g. `epmloaner622.first.last@oracle.com`).


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import http_basic_session
from oracle_ai_data_platform_connectors.rest.epm import (
    list_applications, export_data_slice, slice_to_long_dataframe,
)

session = http_basic_session(
    username=os.environ['EPM_USERNAME'],
    password=os.environ['EPM_PASSWORD'],
    base_url=os.environ['EPM_BASE_URL'],
)
apps = list_applications(session, os.environ['EPM_BASE_URL'])
print('apps:', [a['name'] for a in apps])


In [ ]:
import json
grid = json.loads(os.environ['EPM_GRID_DEFINITION_JSON'])
resp = export_data_slice(session, os.environ['EPM_BASE_URL'], os.environ['EPM_APPLICATION'], os.environ['EPM_PLAN_TYPE'], grid)
df = slice_to_long_dataframe(spark, resp)
df.show(10)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-epm-cloud',
    'auth': 'basic',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
